# 02 — Padrões de Falha

**Objetivo:** entender o *mecanismo* por trás da fricção detectada na EDA.  
Foco: sub-fluxos `shopping_cart` e `credit_card` (os dois primeiros do ICE preview).

**Perguntas:**
1. O que o agente *faz* antes de pedir um try-again? Há diagnóstico ou é trial-and-error direto?
2. Quando o agente faz uma pergunta diagnóstica antes de sugerir solução, o resultado muda?
3. Qual a sequência de ações mais comum? Há um padrão estrutural de falha?
4. O `credit_card` oferece um experimento natural que valida a hipótese para o A/B?

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from pathlib import Path

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW_DIR     = Path('../data/raw')
PROC_DIR    = Path('../data/processed')
FIGURES_DIR = Path('../reports/figures')

with open(RAW_DIR / 'abcd_v1.1.json', 'r', encoding='utf-8') as f:
    raw = json.load(f)

all_convos = [c for split, convos in raw.items() for c in convos]
print(f"Total conversas: {len(all_convos):,}")

## 1. Parsing dos Subfluxos-Alvo

In [ ]:
TARGET_SUBFLOWS = ('shopping_cart', 'credit_card')

# Palavras-chave de diagnóstico: perguntas que identificam a causa ANTES de sugerir solução
DIAG_KEYWORDS = [
    'browser', 'device', 'mobile', 'desktop', 'cache',
    'expir', 'expiration', 'typo', 'different', 'which',
    'what card', 'what browser', 'date', 'digit'
]

records = []
for c in all_convos:
    sf = c['scenario']['subflow']
    if sf not in TARGET_SUBFLOWS:
        continue

    turns      = c['original']
    all_actions = [t['targets'][2] for t in c['delexed']
                   if t['targets'][1] == 'take_action']

    agent_turns  = [(i, t[1]) for i, t in enumerate(turns) if t[0] == 'agent']
    cust_turns   = [(i, t[1]) for i, t in enumerate(turns) if t[0] == 'customer']

    # Pergunta diagnóstica: nos primeiros 4 turnos do agente, há keyword + '?'
    early_agent_text = [t[1].lower() for t in agent_turns[:4]]
    has_diag = any(
        any(kw in text for kw in DIAG_KEYWORDS) and '?' in text
        for text in early_agent_text
    )

    n_try_again  = all_actions.count('try-again')
    n_logout     = all_actions.count('log-out-in')
    action_seq   = tuple(all_actions)

    records.append({
        'convo_id'         : c['convo_id'],
        'subflow'          : sf,
        'n_turns'          : len(turns),
        'n_try_again'      : n_try_again,
        'n_logout'         : n_logout,
        'n_actions'        : len(all_actions),
        'action_seq'       : action_seq,
        'has_diag_question': int(has_diag),
        'resolved_1st_try' : int(n_try_again == 0),
    })

df = pd.DataFrame(records)
print(f"Subfluxos-alvo: {df.groupby('subflow')['convo_id'].count().to_dict()}")
df.head(3)

## 2. Distribuição de Ciclos de Try-Again

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, sf in enumerate(TARGET_SUBFLOWS):
    s = df[df['subflow'] == sf]['n_try_again'].value_counts().sort_index()
    pct = (s / s.sum() * 100).round(1)
    colors = ['#2ca02c' if k == 0 else '#d62728' for k in pct.index]
    bars = axes[i].bar(pct.index.astype(str), pct.values, color=colors, edgecolor='white')
    for bar, val in zip(bars, pct.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                     f'{val:.0f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Número de ciclos try-again')
    axes[i].set_ylabel('% das conversas')
    axes[i].set_title(f'{sf}\n(n={df[df["subflow"]==sf].shape[0]:,})',
                      fontweight='bold')
    green_patch = mpatches.Patch(color='#2ca02c', label='Resolveu de 1ª')
    red_patch   = mpatches.Patch(color='#d62728', label='Precisou retry')
    axes[i].legend(handles=[green_patch, red_patch])

plt.suptitle('Ciclos de Try-Again por Subfluxo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_try_again_distribution.png', bbox_inches='tight')
plt.show()

## 3. Sequências de Ação Mais Comuns

Mapeamos as sequências de ações para identificar o padrão estrutural dominante.

In [ ]:
for sf in TARGET_SUBFLOWS:
    print(f"\n=== {sf}: top 8 sequências ===")
    seq_counts = Counter(df[df['subflow'] == sf]['action_seq'])
    total = sum(seq_counts.values())
    for seq, n in seq_counts.most_common(8):
        has_ta = 'try-again' in seq
        marker = '⚠ ' if has_ta else '✓ '
        print(f"  {marker}{n:3d} ({n/total:.0%})  →  {' → '.join(seq)}")

## 4. Padrão Estrutural: Trial-and-Error sem Diagnóstico

Identificamos se o agente faz perguntas diagnósticas **antes** de sugerir a primeira solução.

In [ ]:
for sf in TARGET_SUBFLOWS:
    s = df[df['subflow'] == sf]
    diag_rate = s['has_diag_question'].mean()
    print(f"{sf}: {diag_rate:.1%} das conversas têm pergunta diagnóstica inicial")
    print(f"  → {1-diag_rate:.1%} vão direto para solução genérica sem diagnóstico")

## 5. Experimento Natural: credit_card

O `credit_card` tem variação natural: 37% dos agentes fazem diagnóstico, 63% não.  
Isso permite comparar os dois grupos como se fosse um A/B não randomizado.

In [ ]:
cc = df[df['subflow'] == 'credit_card'].copy()

cc_groups = cc.groupby('has_diag_question').agg(
    n=('convo_id', 'count'),
    resolved_1st_try=('resolved_1st_try', 'mean'),
    n_try_again_mean=('n_try_again', 'mean'),
    n_turns_mean=('n_turns', 'mean')
).reset_index()
cc_groups['resolved_1st_try'] = (cc_groups['resolved_1st_try'] * 100).round(1)
cc_groups['n_try_again_mean'] = cc_groups['n_try_again_mean'].round(2)
cc_groups['n_turns_mean']     = cc_groups['n_turns_mean'].round(1)
cc_groups['grupo'] = cc_groups['has_diag_question'].map({0: 'Sem diagnóstico (Controle)', 1: 'Com diagnóstico (Variante)'})
print(cc_groups[['grupo', 'n', 'resolved_1st_try', 'n_try_again_mean', 'n_turns_mean']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = [
    ('resolved_1st_try',    'Resolve de 1ª tentativa (%)',   ['#aec7e8', '#2ca02c']),
    ('n_try_again_mean',    'Média de ciclos try-again',     ['#d62728', '#ffbb78']),
    ('n_turns_mean',        'Média de turnos',               ['#d62728', '#ffbb78']),
]

labels = ['Sem\ndiagnóstico', 'Com\ndiagnóstico']

for ax, (col, title, colors) in zip(axes, metrics):
    vals = cc_groups[col].values
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f'{val:.1f}', ha='center', va='bottom', fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(0, vals.max() * 1.25)

plt.suptitle('credit_card — Experimento Natural: Diagnóstico vs Sem Diagnóstico',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_natural_experiment_creditcard.png', bbox_inches='tight')
plt.show()

# Impacto calculado
base  = cc_groups[cc_groups['has_diag_question']==0]['resolved_1st_try'].values[0]
treat = cc_groups[cc_groups['has_diag_question']==1]['resolved_1st_try'].values[0]
print(f"\nEfeito observado (não randomizado): +{treat-base:.1f}pp na resolução de 1ª tentativa")
print(f"Turnos economizados por conversa: {cc_groups['n_turns_mean'].diff().abs().values[1]:.1f}")

## 6. Hipótese Mecanística para shopping_cart

Baseado na evidência do `credit_card`, construímos a hipótese causal para `shopping_cart`.

In [ ]:
sc = df[df['subflow'] == 'shopping_cart']

print("=== shopping_cart: Linha de base ===")
print(f"  Conversas totais:               {len(sc):,}")
print(f"  Resolveu de 1ª (sem try-again): {sc['resolved_1st_try'].mean():.1%}")
print(f"  Precisou de try-again:          {(1-sc['resolved_1st_try']).mean():.1%}")
print(f"  Média de turnos:                {sc['n_turns'].mean():.1f}")
print(f"  Fez pergunta diagnóstica:       {sc['has_diag_question'].mean():.1%}")

print()
print("=== Projeção do efeito (baseado no experimento natural do credit_card) ===")
baseline_sc  = sc['resolved_1st_try'].mean()
effect_cc    = (treat - base) / 100   # +22pp do credit_card
projected_sc = min(baseline_sc + effect_cc * 0.8, 1.0)  # conservador: 80% do efeito observado
print(f"  Baseline shopping_cart:      {baseline_sc:.1%}")
print(f"  Efeito observado (cc):       +{effect_cc:.0%}")
print(f"  Projeção conservadora (80%): {projected_sc:.1%}")
print(f"  MDE para o A/B:              +{projected_sc - baseline_sc:.0%} pp")

## 7. Exemplos Reais de Conversas

Ilustração qualitativa do padrão identificado.

In [ ]:
print("=" * 70)
print("PADRÃO A — Trial-and-Error sem diagnóstico (shopping_cart típico)")
print("=" * 70)

# Encontrar exemplo com try-again=1, sem diagnóstico
for c in all_convos:
    if c['scenario']['subflow'] != 'shopping_cart':
        continue
    actions = [t['targets'][2] for t in c['delexed'] if t['targets'][1] == 'take_action']
    if actions.count('try-again') == 1 and 'log-out-in' in actions:
        for turn in c['original']:
            if turn[0] in ('customer', 'agent'):
                print(f"[{turn[0].upper():8s}] {turn[1]}")
        break

print()
print("=" * 70)
print("PADRÃO B — Com diagnóstico (credit_card com pergunta antes da solução)")
print("=" * 70)

DIAG_KW = ['expir', 'date', 'typo', 'digit', 'browser', 'cache']
for c in all_convos:
    if c['scenario']['subflow'] != 'credit_card':
        continue
    actions = [t['targets'][2] for t in c['delexed'] if t['targets'][1] == 'take_action']
    agent_turns = [t[1].lower() for t in c['original'] if t[0] == 'agent']
    early = agent_turns[:4]
    has_diag = any(any(kw in t for kw in DIAG_KW) and '?' in t for t in early)
    if has_diag and actions.count('try-again') == 0:
        for turn in c['original']:
            if turn[0] in ('customer', 'agent'):
                print(f"[{turn[0].upper():8s}] {turn[1]}")
        break

## 8. Sumário do Notebook 02

### Achados

| Indicador | shopping_cart | credit_card |
|---|---|---|
| Volume | 251 | 266 |
| Taxa de try-again | **84%** | **79%** |
| Resolve de 1ª tentativa | 16% | 21% |
| Tem pergunta diagnóstica | **3%** | 37% |

### Experimento Natural (credit_card)

| Grupo | n | Resolve de 1ª | Try-again médio | Turnos médios |
|---|---|---|---|---|
| Sem diagnóstico | 167 | **13%** | 0.90 | 20.8 |
| Com diagnóstico | 99 | **35%** | 0.65 | 18.7 |
| **Diferença** | | **+22pp** | **-0.25** | **-2.1** |

### Hipótese para o A/B

> **Se** o bot faz 1 pergunta diagnóstica antes de sugerir solução em `shopping_cart`  
> **então** a proporção de conversas sem try-again sobe de ~16% para ~32%  
> **porque** muitas falhas de carrinho são browser-específicas — uma pergunta direciona a solução correta de imediato, eliminando o ciclo de tentativa-e-erro.

**Evidência de suporte:** o experimento natural em `credit_card` mostra efeito de +22pp quando diagnóstico é feito — `shopping_cart` tem zero diagnóstico hoje, com potencial similar.

**Próximo passo:** notebook 03 — métricas de sentimento para enriquecer o diagnóstico e confirmar o impacto na experiência do cliente.